# START

In [1]:
# -*- coding: utf-8 -*-
"""
Kvasir-SEG Thesis Pipeline
DINOv3 + Cosine Similarity + Louvain Pruning + PVT-CASCADE

Colab-ready version of the user's UNet++ thesis implementation.
Main change:
    UNet++  ->  PVT-CASCADE

The pruning logic, train/test split, full-vs-pruned experiments,
BCE/structure-style loss, and Dice/IoU evaluation are kept aligned
with the original thesis implementation as much as possible.
"""

"\nKvasir-SEG Thesis Pipeline\nDINOv3 + Cosine Similarity + Louvain Pruning + PVT-CASCADE\n\nColab-ready version of the user's UNet++ thesis implementation.\nMain change:\n    UNet++  ->  PVT-CASCADE\n\nThe pruning logic, train/test split, full-vs-pruned experiments,\nBCE/structure-style loss, and Dice/IoU evaluation are kept aligned\nwith the original thesis implementation as much as possible.\n"

# 0. COLAB INSTALLATION

In [2]:
!pip install -q kagglehub huggingface_hub timm opencv-python scikit-image networkx python-louvain ml-collections

# Clone official CASCADE repository
import os
import sys
import subprocess
from pathlib import Path

CASCADE_DIR = "/content/CASCADE"

if not os.path.exists(CASCADE_DIR):
    !git clone -q https://github.com/SLDGroup/CASCADE.git /content/CASCADE

# IMPORTANT:
# PVT_CASCADE internally loads:
# ./pretrained_pth/pvt/pvt_v2_b2.pth
# Therefore the working directory must be the CASCADE repository.
os.chdir(CASCADE_DIR)

# Download official PVTv2-B2 ImageNet pretrained weights.
os.makedirs("/content/CASCADE/pretrained_pth/pvt", exist_ok=True)

PVT_WEIGHT = "/content/CASCADE/pretrained_pth/pvt/pvt_v2_b2.pth"

if not os.path.exists(PVT_WEIGHT):
    !wget -q -O /content/CASCADE/pretrained_pth/pvt/pvt_v2_b2.pth \
        https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b2.pth

sys.path.insert(0, CASCADE_DIR)

print("CASCADE directory:", os.getcwd())
print("PVT pretrained weights:", os.path.exists(PVT_WEIGHT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 6.4 MB/s eta 0:00:00
CASCADE directory: /content/CASCADE
PVT pretrained weights: True


# 1. IMPORTS

In [3]:
# ============================================================
import os
import time
import random
import numpy as np
import networkx as nx

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import transforms

from community import community_louvain

from transformers import AutoImageProcessor, AutoModel
import kagglehub

from lib.networks import PVT_CASCADE

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/CASCADE/lib/pvtv2.py:387: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/CASCADE/lib/pvtv2.py:397: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing name. Please check if this is not expect

# 2. CONFIGURATION

In [4]:
# ============================================================
class Config:
    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------
    TRAIN_SPLIT = 0.80
    RANDOM_SEED = 42

    # --------------------------------------------------------
    # PVT-CASCADE
    # --------------------------------------------------------
    IMAGE_SIZE = 224
    BATCH_SIZE = 16
    NUM_WORKERS = 2
    PIN_MEMORY = True

    NUM_CLASSES = 1

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    EPOCHS = 30              # Change to 50/100/200 for final thesis runs
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4

    # --------------------------------------------------------
    # DINOv3 pruning
    # --------------------------------------------------------
    DINO_MODEL_NAME = "rA9del/dinov3b16"

    # Cosine similarity threshold.
    # NOTE: this is cosine similarity directly, not remapped to [0,1].
    SIMILARITY_THRESHOLD = 0.92

    # Fraction selected inside each Louvain community.
    RETENTION_RATIO = 0.45

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------
    THRESHOLD = 0.50

    # --------------------------------------------------------
    # Saving
    # --------------------------------------------------------
    OUTPUT_DIR = "/content/thesis_pvt_cascade_outputs"
    MODEL_DIR = "/content/thesis_pvt_cascade_outputs/checkpoints"

    BEST_FULL_MODEL = "best_pvt_cascade_full.pth"
    BEST_PRUNED_MODEL = "best_pvt_cascade_pruned.pth"

    FEATURE_FILE = "/content/thesis_pvt_cascade_outputs/dinov3_features.npy"
    PRUNED_INDEX_FILE = "/content/thesis_pvt_cascade_outputs/pruned_indices.npy"
    SPLIT_FILE = "/content/thesis_pvt_cascade_outputs/train_test_split.npz"


config = Config()

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)

# 3. REPRODUCIBILITY

In [5]:
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducible behavior.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(config.RANDOM_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("================================================")
print("DEVICE")
print("================================================")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE
Device: cuda
GPU: Tesla T4


# 4. DOWNLOAD CVC-CLINICDB

In [6]:
# ============================================================
print("\n================================================")
print("DOWNLOADING CVC-CLINICDB")
print("================================================")

dataset_path = kagglehub.dataset_download("balraj98/cvcclinicdb")

print("Dataset path:")
print(dataset_path)

# ------------------------------------------------------------
# Automatically find CVC-ClinicDB image and mask directories
# ------------------------------------------------------------

IMAGE_DIR = None
MASK_DIR = None

for root, dirs, files in os.walk(dataset_path):

    folder_name = os.path.basename(root).lower()

    # Look for Original image folder
    if folder_name == "original":
        IMAGE_DIR = root

    # Look for Ground Truth mask folder
    elif folder_name in ["ground truth", "ground_truth", "groundtruth"]:
        MASK_DIR = root

if IMAGE_DIR is None:
    raise FileNotFoundError(
        f"Could not find the CVC-ClinicDB image directory inside: {dataset_path}"
    )

if MASK_DIR is None:
    raise FileNotFoundError(
        f"Could not find the CVC-ClinicDB mask directory inside: {dataset_path}"
    )

print("Image directory:", IMAGE_DIR)
print("Mask directory :", MASK_DIR)

# ------------------------------------------------------------
# Verify that files actually exist
# ------------------------------------------------------------

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
]

mask_files = [
    f for f in os.listdir(MASK_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
]

print("Number of images:", len(image_files))
print("Number of masks :", len(mask_files))

if len(image_files) == 0:
    raise FileNotFoundError(f"No image files found in: {IMAGE_DIR}")

if len(mask_files) == 0:
    raise FileNotFoundError(f"No mask files found in: {MASK_DIR}")


DOWNLOADING CVC-CLINICDB


100%|██████████| 131M/131M [00:01<00:00, 91.5MB/s]

Extracting files...


Dataset path:
/root/.cache/kagglehub/datasets/balraj98/cvcclinicdb/versions/1
Image directory: /root/.cache/kagglehub/datasets/balraj98/cvcclinicdb/versions/1/PNG/Original
Mask directory : /root/.cache/kagglehub/datasets/balraj98/cvcclinicdb/versions/1/PNG/Ground Truth
Number of images: 612
Number of masks : 612


# 5. FIND IMAGE/MASK PAIRS

In [7]:
# ============================================================
def prepare_kvasir_data(image_dir, mask_dir):

    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    mask_files = sorted([
        f for f in os.listdir(mask_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    mask_set = set(mask_files)

    # Keep only images that have a mask with exactly the same filename.
    image_files = [
        f for f in image_files
        if f in mask_set
    ]

    if len(image_files) == 0:
        raise RuntimeError("No matching image/mask pairs found.")

    print(f"Matched image/mask pairs: {len(image_files)}")

    return image_files


all_images = prepare_kvasir_data(IMAGE_DIR, MASK_DIR)

Matched image/mask pairs: 612


# 6. DATASET

In [8]:
class KvasirSegDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        image_names,
        image_size=352
    ):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_names = list(image_names)
        self.image_size = image_size

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        mask_path = os.path.join(
            self.mask_dir,
            image_name
        )

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # PVT-CASCADE is tested here at 352x352.
        image = image.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )

        # IMPORTANT:
        # Never use bilinear/bicubic interpolation for a binary mask.
        mask = mask.resize(
            (self.image_size, self.image_size),
            Image.Resampling.NEAREST
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            mask,
            dtype=np.uint8
        )

        mask = (mask > 127).astype(np.float32)

        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0).float()

        return image, mask


base_dataset = KvasirSegDataset(
    IMAGE_DIR,
    MASK_DIR,
    all_images,
    image_size=config.IMAGE_SIZE
)

print("Total images:", len(base_dataset))

Total images: 612


# 7. TRAIN / TEST SPLIT

In [9]:
# ============================================================
# 7. TRAIN / TEST SPLIT

In [10]:
# ============================================================
# This intentionally follows the original thesis implementation:
# 80% training / 20% testing.
#
# The test set is NEVER passed to the pruning algorithm.

indices = np.arange(len(base_dataset))

rng = np.random.default_rng(config.RANDOM_SEED)
rng.shuffle(indices)

split = int(
    config.TRAIN_SPLIT * len(base_dataset)
)

train_indices = indices[:split].tolist()
test_indices = indices[split:].tolist()

print("\n================================================")
print("TRAIN / TEST SPLIT")
print("================================================")
print("Training images:", len(train_indices))
print("Testing images :", len(test_indices))

print(
    "Overlap:",
    len(set(train_indices) & set(test_indices))
)

# Save split so future experiments can reuse exactly the same split.
np.savez(
    config.SPLIT_FILE,
    train_indices=np.asarray(train_indices),
    test_indices=np.asarray(test_indices)
)


TRAIN / TEST SPLIT
Training images: 489
Testing images : 123
Overlap: 0


# 8. DINOv3 GRAPH PRUNER

In [11]:
# class AccuracyOptimizedGraphPruner:

#     def __init__(
#         self,
#         dataset,
#         train_indices,
#         device
#     ):

#         self.dataset = dataset
#         self.train_indices = train_indices
#         self.device = device

#         print("\n================================================")
#         print("LOADING DINOv3")
#         print("================================================")

#         self.processor = AutoImageProcessor.from_pretrained(
#             config.DINO_MODEL_NAME
#         )

#         self.model = AutoModel.from_pretrained(
#             config.DINO_MODEL_NAME
#         ).to(device)

#         self.model.eval()

#         for param in self.model.parameters():
#             param.requires_grad = False

#         print("DINOv3 loaded and frozen.")

#     @torch.no_grad()
#     def extract_embeddings(self):

#         embeddings = []

#         print("\nExtracting DINOv3 CLS embeddings...")

#         for count, idx in enumerate(self.train_indices):

#             image, _ = self.dataset[idx]

#             # Dataset image is [3,H,W] in [0,1].
#             # Convert directly back to PIL.
#             image_pil = transforms.ToPILImage()(image)

#             inputs = self.processor(
#                 images=image_pil,
#                 return_tensors="pt"
#             )

#             inputs = {
#                 k: v.to(self.device)
#                 for k, v in inputs.items()
#             }

#             outputs = self.model(**inputs)

#             # Global CLS token representation.
#             cls_embedding = outputs.last_hidden_state[:, 0, :]

#             # L2 normalization -> dot product becomes cosine similarity.
#             cls_embedding = F.normalize(
#                 cls_embedding,
#                 p=2,
#                 dim=1
#             )

#             embeddings.append(
#                 cls_embedding.cpu().numpy().flatten()
#             )

#             if (count + 1) % 100 == 0:
#                 print(
#                     f"Processed {count + 1}/"
#                     f"{len(self.train_indices)}"
#                 )

#         embeddings = np.asarray(
#             embeddings,
#             dtype=np.float32
#         )

#         print(
#             "Embedding matrix:",
#             embeddings.shape
#         )

#         return embeddings

#     def prune(
#         self,
#         tau=0.75,
#         p=0.50
#     ):

#         embeddings = self.extract_embeddings()

#         # Save embeddings for later thesis analysis.
#         np.save(
#             config.FEATURE_FILE,
#             embeddings
#         )

#         print("\n================================================")
#         print("COSINE SIMILARITY GRAPH")
#         print("================================================")

#         # Since embeddings are L2 normalized:
#         # dot product = cosine similarity.
#         cosine_sim = np.matmul(
#             embeddings,
#             embeddings.T
#         )

#         cosine_sim = np.clip(
#             cosine_sim,
#             -1.0,
#             1.0
#         )

#         print(
#             "Similarity range:",
#             float(cosine_sim.min()),
#             "to",
#             float(cosine_sim.max())
#         )

#         # Threshold graph.
#         binary_edges = (
#             cosine_sim >= tau
#         ).astype(np.int8)

#         # Remove self-loops.
#         np.fill_diagonal(
#             binary_edges,
#             0
#         )

#         G = nx.from_numpy_array(
#             binary_edges
#         )

#         print("Nodes:", G.number_of_nodes())
#         print("Edges:", G.number_of_edges())

#         # ----------------------------------------------------
#         # Louvain
#         # ----------------------------------------------------

#         print("\nRunning Louvain community detection...")

#         partition = community_louvain.best_partition(
#             G,
#             random_state=config.RANDOM_SEED
#         )

#         communities = {}

#         for node, community_id in partition.items():

#             communities.setdefault(
#                 community_id,
#                 []
#             ).append(node)

#         print(
#             "Number of communities:",
#             len(communities)
#         )

#         sizes = [
#             len(nodes)
#             for nodes in communities.values()
#         ]

#         print("Minimum community size :", min(sizes))
#         print("Maximum community size :", max(sizes))
#         print("Average community size :", np.mean(sizes))

#         # ----------------------------------------------------
#         # Degree-based representative selection
#         # ----------------------------------------------------

#         selected_local_nodes = []

#         for community_id, nodes in communities.items():

#             if len(nodes) <= 1:

#                 selected_local_nodes.append(
#                     nodes[0]
#                 )

#                 continue

#             # Same representative strategy as the original thesis:
#             # highest graph degree inside the Louvain community.
#             degrees = dict(
#                 G.degree(nodes)
#             )

#             sorted_nodes = sorted(
#                 nodes,
#                 key=lambda n: degrees[n],
#                 reverse=True
#             )

#             budget = max(
#                 1,
#                 int(np.ceil(p * len(nodes)))
#             )

#             selected_local_nodes.extend(
#                 sorted_nodes[:budget]
#             )

#         # Map local graph nodes back to dataset indices.
#         pruned_global_indices = [
#             self.train_indices[i]
#             for i in selected_local_nodes
#         ]

#         np.save(
#             config.PRUNED_INDEX_FILE,
#             np.asarray(
#                 pruned_global_indices,
#                 dtype=np.int64
#             )
#         )

#         retention = (
#             len(pruned_global_indices)
#             / len(self.train_indices)
#         )

#         print("\n================================================")
#         print("PRUNING RESULT")
#         print("================================================")
#         print(
#             "Original training samples:",
#             len(self.train_indices)
#         )
#         print(
#             "Selected training samples:",
#             len(pruned_global_indices)
#         )
#         print(
#             f"Actual retention: {retention * 100:.2f}%"
#         )

#         return pruned_global_indices

## New

In [12]:
class AccuracyOptimizedGraphPruner:

    def __init__(
        self,
        dataset,
        train_indices,
        device
    ):

        self.dataset = dataset
        self.train_indices = train_indices
        self.device = device

        # ----------------------------------------------------
        # Read pruning parameters from config
        # ----------------------------------------------------

        self.tau = config.SIMILARITY_THRESHOLD
        self.p = config.RETENTION_RATIO

        print("\n================================================")
        print("LOADING DINOv3")
        print("================================================")

        print(f"Similarity threshold (tau): {self.tau}")
        print(f"Community retention ratio (p): {self.p}")

        if not (0.0 < self.tau <= 1.0):
            raise ValueError(
                f"TAU must be in (0, 1], got {self.tau}"
            )

        if not (0.0 < self.p <= 1.0):
            raise ValueError(
                f"P must be in (0, 1], got {self.p}"
            )

        self.processor = AutoImageProcessor.from_pretrained(
            config.DINO_MODEL_NAME
        )

        self.model = AutoModel.from_pretrained(
            config.DINO_MODEL_NAME
        ).to(self.device)

        self.model.eval()

        for param in self.model.parameters():
            param.requires_grad = False

        print("DINOv3 loaded and frozen.")

    # ========================================================
    # DINOv3 EMBEDDING EXTRACTION
    # ========================================================

    @torch.no_grad()
    def extract_embeddings(self):

        embeddings = []

        print("\n================================================")
        print("EXTRACTING DINOv3 CLS EMBEDDINGS")
        print("================================================")

        for count, idx in enumerate(self.train_indices):

            image, _ = self.dataset[idx]

            # Dataset image must be [3, H, W] in [0, 1].
            image_pil = transforms.ToPILImage()(image)

            inputs = self.processor(
                images=image_pil,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(self.device)
                for key, value in inputs.items()
            }

            outputs = self.model(**inputs)

            # CLS token representation
            cls_embedding = outputs.last_hidden_state[:, 0, :]

            # L2 normalization
            # After normalization:
            # dot product == cosine similarity
            cls_embedding = F.normalize(
                cls_embedding,
                p=2,
                dim=1
            )

            embeddings.append(
                cls_embedding.squeeze(0).cpu().numpy()
            )

            if (count + 1) % 100 == 0:
                print(
                    f"Processed {count + 1}/"
                    f"{len(self.train_indices)}"
                )

        embeddings = np.asarray(
            embeddings,
            dtype=np.float32
        )

        print("\nEmbedding matrix shape:")
        print(embeddings.shape)

        return embeddings

    # ========================================================
    # GRAPH CONSTRUCTION
    # ========================================================

    def build_similarity_graph(self, embeddings):

        print("\n================================================")
        print("BUILDING COSINE SIMILARITY GRAPH")
        print("================================================")

        # Because embeddings are L2 normalized:
        #
        # cosine(x_i, x_j)
        # =
        # x_i dot x_j
        #
        cosine_sim = np.matmul(
            embeddings,
            embeddings.T
        )

        cosine_sim = np.clip(
            cosine_sim,
            -1.0,
            1.0
        )

        print(
            "Similarity range:",
            float(cosine_sim.min()),
            "to",
            float(cosine_sim.max())
        )

        # ----------------------------------------------------
        # Threshold graph
        # ----------------------------------------------------

        binary_edges = (
            cosine_sim >= self.tau
        ).astype(np.int8)

        # Remove self-loops
        np.fill_diagonal(
            binary_edges,
            0
        )

        G = nx.from_numpy_array(
            binary_edges
        )

        num_nodes = G.number_of_nodes()
        num_edges = G.number_of_edges()

        # Maximum possible number of undirected edges
        max_edges = (
            num_nodes * (num_nodes - 1)
        ) / 2

        graph_density = (
            num_edges / max_edges
            if max_edges > 0
            else 0.0
        )

        degrees = [
            degree
            for _, degree in G.degree()
        ]

        isolated_nodes = sum(
            degree == 0
            for degree in degrees
        )

        print("\nGraph statistics:")
        print("Nodes              :", num_nodes)
        print("Edges              :", num_edges)
        print(
            f"Graph density      : {graph_density:.6f}"
        )
        print(
            f"Average degree     : {np.mean(degrees):.4f}"
        )
        print(
            f"Maximum degree     : {np.max(degrees)}"
        )
        print(
            f"Isolated nodes     : {isolated_nodes}"
        )

        return G

    # ========================================================
    # LOUVAIN COMMUNITY DETECTION
    # ========================================================

    def detect_communities(self, G):

        print("\n================================================")
        print("RUNNING LOUVAIN COMMUNITY DETECTION")
        print("================================================")

        partition = community_louvain.best_partition(
            G,
            random_state=config.RANDOM_SEED
        )

        communities = {}

        for node, community_id in partition.items():

            communities.setdefault(
                community_id,
                []
            ).append(node)

        sizes = [
            len(nodes)
            for nodes in communities.values()
        ]

        print(
            "Number of communities:",
            len(communities)
        )

        print(
            "Minimum community size :",
            min(sizes)
        )

        print(
            "Maximum community size :",
            max(sizes)
        )

        print(
            "Average community size :",
            np.mean(sizes)
        )

        print(
            "Singleton communities  :",
            sum(size == 1 for size in sizes)
        )

        return communities

    # ========================================================
    # REPRESENTATIVE SELECTION
    # ========================================================

    def select_representatives(
        self,
        G,
        communities
    ):

        print("\n================================================")
        print("SELECTING COMMUNITY REPRESENTATIVES")
        print("================================================")

        selected_local_nodes = []

        community_statistics = []

        for community_id, nodes in communities.items():

            community_size = len(nodes)

            # ------------------------------------------------
            # Singleton community
            # ------------------------------------------------

            if community_size == 1:

                selected_local_nodes.append(
                    nodes[0]
                )

                community_statistics.append({
                    "community_id": community_id,
                    "size": 1,
                    "selected": 1,
                    "retention": 1.0
                })

                continue

            # ------------------------------------------------
            # IMPORTANT FIX:
            #
            # Calculate degree ONLY inside this community.
            # ------------------------------------------------

            subgraph = G.subgraph(nodes)

            degrees = dict(
                subgraph.degree()
            )

            # ------------------------------------------------
            # Number of samples to retain
            # ------------------------------------------------

            budget = max(
                1,
                int(
                    np.ceil(
                        self.p * community_size
                    )
                )
            )

            # ------------------------------------------------
            # Sort:
            #
            # 1. Highest intra-community degree
            # 2. Lowest node index for deterministic
            #    tie-breaking
            # ------------------------------------------------

            sorted_nodes = sorted(
                nodes,
                key=lambda node: (
                    -degrees[node],
                    node
                )
            )

            selected_nodes = sorted_nodes[:budget]

            selected_local_nodes.extend(
                selected_nodes
            )

            community_statistics.append({
                "community_id": community_id,
                "size": community_size,
                "selected": budget,
                "retention": budget / community_size
            })

        # ----------------------------------------------------
        # Sort final selected nodes for reproducibility
        # ----------------------------------------------------

        selected_local_nodes = sorted(
            selected_local_nodes
        )

        print(
            "Selected local nodes:",
            len(selected_local_nodes)
        )

        return (
            selected_local_nodes,
            community_statistics
        )

    # ========================================================
    # COMPLETE PRUNING PIPELINE
    # ========================================================

    def prune(self):

        # ----------------------------------------------------
        # 1. Extract DINOv3 embeddings
        # ----------------------------------------------------

        embeddings = self.extract_embeddings()

        # Save embeddings for thesis analysis
        np.save(
            config.FEATURE_FILE,
            embeddings
        )

        # ----------------------------------------------------
        # 2. Build similarity graph
        # ----------------------------------------------------

        G = self.build_similarity_graph(
            embeddings
        )

        # ----------------------------------------------------
        # 3. Louvain community detection
        # ----------------------------------------------------

        communities = self.detect_communities(
            G
        )

        # ----------------------------------------------------
        # 4. Select representatives
        # ----------------------------------------------------

        (
            selected_local_nodes,
            community_statistics
        ) = self.select_representatives(
            G,
            communities
        )

        # ----------------------------------------------------
        # 5. Map local graph nodes to dataset indices
        # ----------------------------------------------------

        pruned_global_indices = [
            self.train_indices[i]
            for i in selected_local_nodes
        ]

        # ----------------------------------------------------
        # 6. Save selected indices
        # ----------------------------------------------------

        np.save(
            config.PRUNED_INDEX_FILE,
            np.asarray(
                pruned_global_indices,
                dtype=np.int64
            )
        )

        # ----------------------------------------------------
        # 7. Calculate actual global retention
        # ----------------------------------------------------

        original_size = len(
            self.train_indices
        )

        selected_size = len(
            pruned_global_indices
        )

        retention = (
            selected_size / original_size
        )

        pruning_rate = 1.0 - retention

        # ----------------------------------------------------
        # 8. Final report
        # ----------------------------------------------------

        print("\n================================================")
        print("PRUNING RESULT")
        print("================================================")

        print(
            "Original training samples:",
            original_size
        )

        print(
            "Selected training samples:",
            selected_size
        )

        print(
            f"Target community retention (p): "
            f"{self.p:.4f}"
        )

        print(
            f"Actual global retention: "
            f"{retention * 100:.2f}%"
        )

        print(
            f"Actual global pruning rate: "
            f"{pruning_rate * 100:.2f}%"
        )

        print(
            f"Similarity threshold (tau): "
            f"{self.tau:.4f}"
        )

        print(
            "Saved embeddings to:",
            config.FEATURE_FILE
        )

        print(
            "Saved pruned indices to:",
            config.PRUNED_INDEX_FILE
        )

        return pruned_global_indices

# 9. RUN PRUNING

In [13]:
# ============================================================
print("\n================================================")
print("DINOv3 + LOUVAIN DATASET PRUNING")
print("================================================")

pruner = AccuracyOptimizedGraphPruner(
    dataset=base_dataset,
    train_indices=train_indices,
    device=device
)

pruned_train_indices = pruner.prune(
    # tau=config.SIMILARITY_THRESHOLD,
    # p=config.RETENTION_RATIO
)


DINOv3 + LOUVAIN DATASET PRUNING

LOADING DINOv3
Similarity threshold (tau): 0.92
Community retention ratio (p): 0.45


preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

DINOv3 loaded and frozen.

EXTRACTING DINOv3 CLS EMBEDDINGS
Processed 100/489
Processed 200/489
Processed 300/489
Processed 400/489

Embedding matrix shape:
(489, 768)

BUILDING COSINE SIMILARITY GRAPH
Similarity range: 0.3912596106529236 to 1.0

Graph statistics:
Nodes              : 489
Edges              : 19328
Graph density      : 0.161990
Average degree     : 79.0511
Maximum degree     : 204
Isolated nodes     : 1

RUNNING LOUVAIN COMMUNITY DETECTION
Number of communities: 5
Minimum community size : 1
Maximum community size : 163
Average community size : 97.8
Singleton communities  : 1

SELECTING COMMUNITY REPRESENTATIVES
Selected local nodes: 223

PRUNING RESULT
Original training samples: 489
Selected training samples: 223
Target community retention (p): 0.4500
Actual global retention: 45.60%
Actual global pruning rate: 54.40%
Similarity threshold (tau): 0.9200
Saved embeddings to: /content/thesis_pvt_cascade_outputs/dinov3_features.npy
Saved pruned indices to: /content/thesis_p

# 10. DATALOADERS

In [14]:
# ============================================================
full_train_loader = DataLoader(
    Subset(
        base_dataset,
        train_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

pruned_train_loader = DataLoader(
    Subset(
        base_dataset,
        pruned_train_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

test_loader = DataLoader(
    Subset(
        base_dataset,
        test_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

print("\nDataLoaders created.")


DataLoaders created.


# 11. PVT-CASCADE STRUCTURE LOSS

In [15]:
class StructureLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, mask):

        # Boundary-aware weighting.
        weit = 1 + 5 * torch.abs(
            F.avg_pool2d(
                mask,
                kernel_size=31,
                stride=1,
                padding=15
            ) - mask
        )

        # Weighted BCE.
        wbce = F.binary_cross_entropy_with_logits(
            pred,
            mask,
            reduction="none"
        )

        wbce = (
            (weit * wbce).sum(dim=(2, 3))
            / weit.sum(dim=(2, 3))
        )

        # Weighted IoU.
        pred_prob = torch.sigmoid(pred)

        inter = (
            pred_prob * mask * weit
        ).sum(dim=(2, 3))

        union = (
            (pred_prob + mask) * weit
        ).sum(dim=(2, 3))

        wiou = 1 - (
            (inter + 1)
            / (union - inter + 1)
        )

        return (
            wbce + wiou
        ).mean()

# 12. PVT-CASCADE MODEL FACTORY

In [16]:
def create_pvt_cascade():

    print("\nCreating PVT-CASCADE...")

    model = PVT_CASCADE(
        n_class=config.NUM_CLASSES
    )

    model = model.to(device)

    return model

# 13. TRAINING FUNCTION

In [17]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        p1, p2, p3, p4 = model(
            images
        )

        loss1 = criterion(
            p1,
            masks
        )

        loss2 = criterion(
            p2,
            masks
        )

        loss3 = criterion(
            p3,
            masks
        )

        loss4 = criterion(
            p4,
            masks
        )

        # Deep supervision.
        loss = (
            loss1 +
            loss2 +
            loss3 +
            loss4
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return (
        total_loss /
        max(1, len(loader))
    )

# 14. EVALUATION

In [18]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    device
):

    model.eval()

    total_dice = 0.0
    total_iou = 0.0
    total_correct = 0
    total_pixels = 0

    n_images = 0

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        p1, p2, p3, p4 = model(
            images
        )

        # Same aggregation strategy as your PVT-CASCADE file.
        logits = (
            p1 +
            p2 +
            p3 +
            p4
        )

        probs = torch.sigmoid(
            logits
        )

        preds = (
            probs > config.THRESHOLD
        ).float()

        intersection = (
            preds * masks
        ).sum(
            dim=(1, 2, 3)
        )

        pred_area = preds.sum(
            dim=(1, 2, 3)
        )

        gt_area = masks.sum(
            dim=(1, 2, 3)
        )

        dice = (
            2 * intersection + 1e-7
        ) / (
            pred_area +
            gt_area +
            1e-7
        )

        union = (
            pred_area +
            gt_area -
            intersection
        )

        iou = (
            intersection + 1e-7
        ) / (
            union + 1e-7
        )

        total_dice += dice.sum().item()
        total_iou += iou.sum().item()

        total_correct += (
            preds == masks
        ).sum().item()

        total_pixels += masks.numel()

        n_images += images.size(0)

    avg_dice = (
        total_dice /
        max(1, n_images)
    )

    avg_iou = (
        total_iou /
        max(1, n_images)
    )

    accuracy = (
        total_correct /
        max(1, total_pixels)
    )

    return (
        avg_dice,
        avg_iou,
        accuracy
    )

# 15. EXPERIMENT RUNNER

In [19]:
def run_experiment(
    experiment_name,
    train_loader,
    checkpoint_name
):

    print("\n")
    print("=" * 60)
    print(f"TRAINING: {experiment_name}")
    print("=" * 60)

    model = create_pvt_cascade()

    criterion = StructureLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )

    best_dice = -1.0

    history = []

    start_time = time.time()

    for epoch in range(
        config.EPOCHS
    ):

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        dice, iou, accuracy = evaluate(
            model,
            test_loader,
            device
        )

        epoch_time = (
            time.time() -
            epoch_start
        )

        history.append({
            "epoch": epoch + 1,
            "loss": train_loss,
            "dice": dice,
            "iou": iou,
            "accuracy": accuracy,
            "epoch_time": epoch_time
        })

        print(
            f"Epoch [{epoch+1}/{config.EPOCHS}] "
            f"Loss: {train_loss:.4f} | "
            f"Dice: {dice:.4f} | "
            f"IoU: {iou:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Time: {epoch_time:.1f}s"
        )

        # Keep the best model based on Dice.
        if dice > best_dice:

            best_dice = dice

            checkpoint_path = os.path.join(
                config.MODEL_DIR,
                checkpoint_name
            )

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch + 1,
                    "dice": dice,
                    "iou": iou,
                    "accuracy": accuracy,
                    "experiment": experiment_name
                },
                checkpoint_path
            )

            print(
                "  -> Saved:",
                checkpoint_path
            )

    total_time = (
        time.time() -
        start_time
    )

    # Load best checkpoint.
    checkpoint_path = os.path.join(
        config.MODEL_DIR,
        checkpoint_name
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    final_dice, final_iou, final_accuracy = evaluate(
        model,
        test_loader,
        device
    )

    return {
        "experiment": experiment_name,
        "training_time": total_time,
        "best_epoch": checkpoint["epoch"],
        "dice": final_dice,
        "iou": final_iou,
        "accuracy": final_accuracy,
        "history": history,
        "checkpoint": checkpoint_path
    }

# 16. EXPERIMENT 1 — FULL TRAINING SET

In [20]:
full_result = run_experiment(
    experiment_name="PVT-CASCADE — FULL TRAINING SET",
    train_loader=full_train_loader,
    checkpoint_name=config.BEST_FULL_MODEL
)



TRAINING: PVT-CASCADE — FULL TRAINING SET

Creating PVT-CASCADE...
Epoch [1/30] Loss: 5.2839 | Dice: 0.6995 | IoU: 0.5759 | Accuracy: 0.9528 | Time: 24.8s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [2/30] Loss: 3.8208 | Dice: 0.8406 | IoU: 0.7479 | Accuracy: 0.9744 | Time: 17.9s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [3/30] Loss: 3.2799 | Dice: 0.8649 | IoU: 0.7822 | Accuracy: 0.9791 | Time: 18.1s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [4/30] Loss: 2.9927 | Dice: 0.8835 | IoU: 0.8084 | Accuracy: 0.9821 | Time: 20.2s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [5/30] Loss: 2.7623 | Dice: 0.8740 | IoU: 0.7894 | Accuracy: 0.9804 | Time: 18.5s
Epoch [6/30] Loss: 2.5860 | Dice: 0.8884 | IoU: 0.8128 | Accuracy: 0.9834 | Time: 17.4s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints

# 17. EXPERIMENT 2 — PRUNED TRAINING SET

In [21]:
pruned_result = run_experiment(
    experiment_name="PVT-CASCADE — PRUNED TRAINING SET",
    train_loader=pruned_train_loader,
    checkpoint_name=config.BEST_PRUNED_MODEL
)



TRAINING: PVT-CASCADE — PRUNED TRAINING SET

Creating PVT-CASCADE...
Epoch [1/30] Loss: 5.8846 | Dice: 0.1627 | IoU: 0.1164 | Accuracy: 0.9199 | Time: 11.2s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [2/30] Loss: 4.7228 | Dice: 0.6720 | IoU: 0.5512 | Accuracy: 0.9510 | Time: 9.3s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [3/30] Loss: 4.0477 | Dice: 0.7185 | IoU: 0.5931 | Accuracy: 0.9526 | Time: 9.0s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [4/30] Loss: 3.6713 | Dice: 0.7750 | IoU: 0.6620 | Accuracy: 0.9626 | Time: 8.7s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [5/30] Loss: 3.3955 | Dice: 0.8027 | IoU: 0.6974 | Accuracy: 0.9666 | Time: 8.6s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [6/30] Loss: 3.1813 | Dice: 0.7932 | IoU: 0.678

# 18. FINAL RESULTS

In [22]:
print("\n")
print("=" * 70)
print("FINAL THESIS RESULTS")
print("=" * 70)

print("\nConfiguration")
print("-" * 70)
print("Image size          :", config.IMAGE_SIZE)
print("Batch size          :", config.BATCH_SIZE)
print("Epochs              :", config.EPOCHS)
print("Learning rate       :", config.LEARNING_RATE)
print("Weight decay        :", config.WEIGHT_DECAY)
print("DINO model          :", config.DINO_MODEL_NAME)
print("Similarity threshold:", config.SIMILARITY_THRESHOLD)
print("Requested retention :", config.RETENTION_RATIO)

print("\nDataset")
print("-" * 70)
print("Total images        :", len(base_dataset))
print("Original train      :", len(train_indices))
print("Pruned train        :", len(pruned_train_indices))
print("Test                :", len(test_indices))

actual_retention = (
    len(pruned_train_indices) /
    len(train_indices)
)

print(
    f"Actual retention    : {actual_retention * 100:.2f}%"
)

print("\nFULL TRAINING SET")
print("-" * 70)
print(
    f"Training time : "
    f"{full_result['training_time']:.2f} sec"
)
print(
    f"Best epoch    : "
    f"{full_result['best_epoch']}"
)
print(
    f"Dice          : "
    f"{full_result['dice']:.4f}"
)
print(
    f"IoU           : "
    f"{full_result['iou']:.4f}"
)
print(
    f"Accuracy      : "
    f"{full_result['accuracy']:.4f}"
)

print("\nPRUNED TRAINING SET")
print("-" * 70)
print(
    f"Training time : "
    f"{pruned_result['training_time']:.2f} sec"
)
print(
    f"Best epoch    : "
    f"{pruned_result['best_epoch']}"
)
print(
    f"Dice          : "
    f"{pruned_result['dice']:.4f}"
)
print(
    f"IoU           : "
    f"{pruned_result['iou']:.4f}"
)
print(
    f"Accuracy      : "
    f"{pruned_result['accuracy']:.4f}"
)

print("\nCOMPARISON")
print("-" * 70)

dice_change = (
    pruned_result["dice"] -
    full_result["dice"]
)

iou_change = (
    pruned_result["iou"] -
    full_result["iou"]
)

training_speedup = (
    full_result["training_time"] /
    max(pruned_result["training_time"], 1e-8)
)

print(
    f"Dice change       : {dice_change:+.4f}"
)

print(
    f"IoU change        : {iou_change:+.4f}"
)

print(
    f"Training speedup  : {training_speedup:.2f}x"
)

print("\nSaved files")
print("-" * 70)
print(
    "DINO features :",
    config.FEATURE_FILE
)

print(
    "Pruned index  :",
    config.PRUNED_INDEX_FILE
)

print(
    "Full model    :",
    full_result["checkpoint"]
)

print(
    "Pruned model  :",
    pruned_result["checkpoint"]
)

print("\nDONE.")



FINAL THESIS RESULTS

Configuration
----------------------------------------------------------------------
Image size          : 224
Batch size          : 16
Epochs              : 30
Learning rate       : 0.0001
Weight decay        : 0.0001
DINO model          : rA9del/dinov3b16
Similarity threshold: 0.92
Requested retention : 0.45

Dataset
----------------------------------------------------------------------
Total images        : 612
Original train      : 489
Pruned train        : 223
Test                : 123
Actual retention    : 45.60%

FULL TRAINING SET
----------------------------------------------------------------------
Training time : 535.48 sec
Best epoch    : 25
Dice          : 0.9289
IoU           : 0.8782
Accuracy      : 0.9885

PRUNED TRAINING SET
----------------------------------------------------------------------
Training time : 273.47 sec
Best epoch    : 28
Dice          : 0.8797
IoU           : 0.8081
Accuracy      : 0.9777

COMPARISON
---------------------------

## FINAL RESULT - PRUNED DATASET

In [23]:
# # ============================================================
# # FINAL THESIS RESULTS REPORT
# # ============================================================
# # IMPORTANT:
# # The FULL DATASET result is FIXED.
# # It has already been trained and therefore does NOT need to
# # be retrained for every retention-ratio experiment.
# #
# # Only the PRUNED DATASET is trained for each experiment.
# # ============================================================


# print("\n")
# print("=" * 70)
# print("FINAL THESIS RESULTS")
# print("=" * 70)


# # ============================================================
# # 1. FIXED FULL-DATASET BASELINE
# # ============================================================
# # These values come from the completed final full-dataset run.
# # DO NOT retrain the full dataset for every retention ratio.

# FULL_DATASET_TOTAL_IMAGES = 1000

# FULL_TRAIN_IMAGES = 800

# FULL_TEST_IMAGES = 200

# FULL_EPOCHS = 50

# FULL_IMAGE_SIZE = 224

# FULL_BATCH_SIZE = 16

# FULL_LEARNING_RATE = 1e-4

# FULL_WEIGHT_DECAY = 1e-4


# # Best full-dataset validation result
# FULL_BEST_EPOCH = 53

# FULL_DICE = 0.8989

# FULL_IOU = 0.8461

# FULL_ACCURACY = 0.9710

# FULL_TRAINING_TIME = 1416.025 # 2832.05(100 epochs)


# # ============================================================
# # 2. CURRENT PRUNED EXPERIMENT
# # ============================================================
# # These values are obtained from the current experiment.

# PRUNED_TRAIN_IMAGES = len(pruned_train_indices)

# PRUNED_EPOCHS = config.EPOCHS

# PRUNED_TRAINING_TIME = pruned_result["training_time"]

# PRUNED_BEST_EPOCH = pruned_result["best_epoch"]

# PRUNED_DICE = pruned_result["dice"]

# PRUNED_IOU = pruned_result["iou"]

# PRUNED_ACCURACY = pruned_result["accuracy"]


# # ============================================================
# # 3. DATASET RETENTION / PRUNING CALCULATIONS
# # ============================================================

# actual_retention = (
#     PRUNED_TRAIN_IMAGES / FULL_TRAIN_IMAGES
# )

# pruning_rate = (
#     1.0 - actual_retention
# )

# images_removed = (
#     FULL_TRAIN_IMAGES - PRUNED_TRAIN_IMAGES
# )


# # ============================================================
# # 4. PERFORMANCE DIFFERENCE
# # ============================================================

# dice_change = (
#     PRUNED_DICE - FULL_DICE
# )

# iou_change = (
#     PRUNED_IOU - FULL_IOU
# )

# accuracy_change = (
#     PRUNED_ACCURACY - FULL_ACCURACY
# )


# # Absolute performance drop
# dice_drop = (
#     FULL_DICE - PRUNED_DICE
# )

# iou_drop = (
#     FULL_IOU - PRUNED_IOU
# )


# # ============================================================
# # 5. TRAINING EFFICIENCY
# # ============================================================

# training_speedup = (
#     FULL_TRAINING_TIME /
#     max(PRUNED_TRAINING_TIME, 1e-8)
# )

# training_time_reduction = (
#     1.0 -
#     PRUNED_TRAINING_TIME / FULL_TRAINING_TIME
# )


# # ============================================================
# # 6. PRINT CONFIGURATION
# # ============================================================

# print("\nCONFIGURATION")
# print("-" * 70)

# print("Image size              :", FULL_IMAGE_SIZE)

# print("Batch size              :", FULL_BATCH_SIZE)

# print("Learning rate           :", FULL_LEARNING_RATE)

# print("Weight decay            :", FULL_WEIGHT_DECAY)

# print("Full dataset epochs     :", FULL_EPOCHS)

# print("Pruned dataset epochs   :", PRUNED_EPOCHS)

# print("DINO model              :", config.DINO_MODEL_NAME)

# print("Similarity threshold    :", config.SIMILARITY_THRESHOLD)

# print("Requested retention     :",
#       f"{config.RETENTION_RATIO * 100:.2f}%")


# # ============================================================
# # 7. DATASET INFORMATION
# # ============================================================

# print("\nDATASET")
# print("-" * 70)

# print("Total images            :", FULL_DATASET_TOTAL_IMAGES)

# print("Full training images    :", FULL_TRAIN_IMAGES)

# print("Pruned training images  :", PRUNED_TRAIN_IMAGES)

# print("Test images             :", FULL_TEST_IMAGES)

# print(
#     f"Actual retention        : "
#     f"{actual_retention * 100:.2f}%"
# )

# print(
#     f"Images removed          : "
#     f"{images_removed}"
# )

# print(
#     f"Pruning rate            : "
#     f"{pruning_rate * 100:.2f}%"
# )


# # ============================================================
# # 8. FIXED FULL DATASET RESULT
# # ============================================================

# print("\nFULL DATASET BASELINE")
# print("-" * 70)

# print(
#     f"Training images : "
#     f"{FULL_TRAIN_IMAGES}"
# )

# print(
#     f"Training time   : "
#     f"{FULL_TRAINING_TIME:.2f} sec"
# )

# print(
#     f"Best epoch      : "
#     f"{FULL_BEST_EPOCH}"
# )

# print(
#     f"Dice            : "
#     f"{FULL_DICE:.4f}"
# )

# print(
#     f"IoU             : "
#     f"{FULL_IOU:.4f}"
# )

# print(
#     f"Accuracy        : "
#     f"{FULL_ACCURACY:.4f}"
# )


# # ============================================================
# # 9. CURRENT PRUNED DATASET RESULT
# # ============================================================

# print("\nPRUNED DATASET")
# print("-" * 70)

# print(
#     f"Training images : "
#     f"{PRUNED_TRAIN_IMAGES}"
# )

# print(
#     f"Training time   : "
#     f"{PRUNED_TRAINING_TIME:.2f} sec"
# )

# print(
#     f"Best epoch      : "
#     f"{PRUNED_BEST_EPOCH}"
# )

# print(
#     f"Dice            : "
#     f"{PRUNED_DICE:.4f}"
# )

# print(
#     f"IoU             : "
#     f"{PRUNED_IOU:.4f}"
# )

# print(
#     f"Accuracy        : "
#     f"{PRUNED_ACCURACY:.4f}"
# )


# # ============================================================
# # 10. COMPARISON
# # ============================================================

# print("\nCOMPARISON WITH FULL DATASET")
# print("-" * 70)

# print(
#     f"Retention               : "
#     f"{actual_retention * 100:.2f}%"
# )

# print(
#     f"Data removed            : "
#     f"{pruning_rate * 100:.2f}%"
# )

# print(
#     f"Dice change             : "
#     f"{dice_change:+.4f}"
# )

# print(
#     f"IoU change              : "
#     f"{iou_change:+.4f}"
# )

# print(
#     f"Accuracy change         : "
#     f"{accuracy_change:+.4f}"
# )

# print(
#     f"Absolute Dice drop      : "
#     f"{dice_drop:.4f}"
# )

# print(
#     f"Absolute IoU drop       : "
#     f"{iou_drop:.4f}"
# )

# print(
#     f"Training speedup        : "
#     f"{training_speedup:.2f}x"
# )

# print(
#     f"Training time reduction : "
#     f"{training_time_reduction * 100:.2f}%"
# )


# # ============================================================
# # 11. THESIS SUMMARY
# # ============================================================

# print("\nTHESIS SUMMARY")
# print("-" * 70)

# print(
#     f"Full dataset: "
#     f"{FULL_TRAIN_IMAGES} images"
# )

# print(
#     f"Pruned dataset: "
#     f"{PRUNED_TRAIN_IMAGES} images"
# )

# print(
#     f"Dataset reduction: "
#     f"{pruning_rate * 100:.2f}%"
# )

# print(
#     f"Full Dice: "
#     f"{FULL_DICE:.4f}"
# )

# print(
#     f"Pruned Dice: "
#     f"{PRUNED_DICE:.4f}"
# )

# print(
#     f"Dice difference: "
#     f"{dice_change:+.4f}"
# )

# print(
#     f"Full IoU: "
#     f"{FULL_IOU:.4f}"
# )

# print(
#     f"Pruned IoU: "
#     f"{PRUNED_IOU:.4f}"
# )

# print(
#     f"IoU difference: "
#     f"{iou_change:+.4f}"
# )

# print(
#     f"Training speedup: "
#     f"{training_speedup:.2f}x"
# )


# # ============================================================
# # 12. SAVED FILES
# # ============================================================

# print("\nSAVED FILES")
# print("-" * 70)

# print(
#     "DINO features        :",
#     config.FEATURE_FILE
# )

# print(
#     "Pruned indices        :",
#     config.PRUNED_INDEX_FILE
# )

# print(
#     "Pruned model          :",
#     pruned_result["checkpoint"]
# )


# # ============================================================
# # 13. FINAL STATUS
# # ============================================================

# print("\n" + "=" * 70)
# print("FULL DATASET BASELINE WAS USED AS A FIXED REFERENCE")
# print("ONLY THE PRUNED DATASET WAS TRAINED IN THIS EXPERIMENT")
# print("=" * 70)

# print("\nDONE.")

# END